In [16]:
# =============================================================================
# DIRECT PREFERENCE OPTIMIZATION (DPO) — preference tuning for Correction-GPT
# =============================================================================
#
# ---------------------------------------------------------------------------
# WHERE ARE WE ON THE PATH? (picture the 3 stages again)
# ---------------------------------------------------------------------------
#   1) PRETRAIN   — read tons of text → learn language (next-token guessing)
#   2) SFT        — flashcards: one ideal answer per prompt (notebook 9)
#   3) PREFERENCE — THIS NOTEBOOK
#        For the SAME prompt, humans (or you) say:
#          "answer A is better than answer B"
#        Model learns to prefer A over B — tone, brevity, confidence, etc.
#
# Easy analogy:
#   SFT  = show the student the answer key once.
#   DPO  = show two essays and circle the better one.
#         "Write more like THIS, less like THAT."
#
# Why not stop at SFT?
#   SFT only shows ONE gold reply. It does NOT teach:
#     - short > rambling
#     - confident correction > "I'm not sure, maybe ask someone..."
#     - whisper style > essay style
#   Preferences fill that gap WITHOUT needing a separate reward model
#   + RL loop (classic RLHF). DPO does preference learning in one loss.
#
#
# ---------------------------------------------------------------------------
# TOPIC: what is DPO? (deep but easy)
# ---------------------------------------------------------------------------
# Classic RLHF (rough cartoon):
#
#   SFT model ──▶ sample answers ──▶ humans rank them
#                      │
#                      ▼
#              train a REWARD model (scores "how good")
#                      │
#                      ▼
#              RL (PPO) nudges policy to get high reward
#              (tricky: unstable, many knobs)
#
# DPO shortcut:
#
#   Keep a frozen REFERENCE model (usually the SFT checkpoint).
#   Train POLICY model so that:
#     preferred (chosen) answer becomes MORE likely than rejected,
#     relative to what the reference already thought.
#
# Sticky intuition (no formulas yet):
#   ┌──────── prompt ────────┐
#   │ Ground truth + user    │
#   └───────────┬────────────┘
#               │
#        ┌──────┴──────┐
#        ▼             ▼
#   CHOSEN ✅      REJECTED ❌
#   short, clear   long, hedgy, unsure
#        │             │
#        └──────┬──────┘
#               ▼
#   "Push probability UP on chosen tokens,
#    DOWN on rejected tokens — but don't
#    wander too far from the SFT reference."
#
# Same Correction-GPT product idea:
#   Earpiece should whisper a CLEAN correction, not a nervous paragraph.
#
#
# ---------------------------------------------------------------------------
# THIS CELL — build a tiny PREFERENCE dataset + train BPE on it
# ---------------------------------------------------------------------------
# Each row has THREE strings:
#   prompt   — the situation (truth + what user said)
#   chosen   — the answer we WANT (concise Correction: ...)
#   rejected — the answer we DON'T WANT (verbose / uncertain)
#
# Later cells will score both completions under the policy + reference
# and apply the DPO loss. Today we only prepare data + tokenizer.
#

from pathlib import Path
import sys
import json

# week2/ on path so `from tokenizer import BPETokenizer` works
_WEEK2 = Path.cwd() / "week2"
if _WEEK2.is_dir():
    sys.path.insert(0, str(_WEEK2.resolve()))
elif Path.cwd().name == "week2":
    sys.path.insert(0, str(Path.cwd().resolve()))
else:
    sys.path.insert(0, str(Path("week2").resolve()))

import importlib
import tokenizer as _tokenizer_mod
importlib.reload(_tokenizer_mod)  # pick up save/load if kernel had an old import
from tokenizer import BPETokenizer  # week2/tokenizer.py (notebook 8)
import torch

# ---------------------------------------------------------------------------
# Preference Dataset (Chosen vs Rejected)
# ---------------------------------------------------------------------------
# Picture for ONE row:
#
#   PROMPT
#   ┌─────────────────────────────────────────────────────────┐
#   │ Ground truth: The price is $500/month                   │
#   │ User said:    It costs $300 a month                     │
#   └─────────────────────────────────────────────────────────┘
#            │                              │
#            ▼ chosen ✅                    ▼ rejected ❌
#   ┌──────────────────────────┐   ┌────────────────────────────────────┐
#   │ Correction: The price is │   │ Well, you may want to double-check │
#   │ $500 per month, not $300.│   │ ... I'm not entirely sure...       │
#   └──────────────────────────┘   └────────────────────────────────────┘
#
# Sticky: SAME prompt, TWO futures. DPO learns which future to prefer.

preference_data = [
    {
        "prompt": "Ground truth: The price is $500/month\nUser said: It costs $300 a month",
        "chosen": "Correction: The price is $500 per month, not $300.",
        "rejected": (
            "Well, you may want to double-check that because I think the actual "
            "price is somewhere around $500, but I'm not entirely sure, let me look it up..."
        ),
    },
    {
        "prompt": "Ground truth: We are SOC 2 compliant\nUser said: Are we SOC 2?",
        "chosen": "Correction: Yes, we are SOC 2 compliant.",
        "rejected": (
            "I believe we are SOC 2 compliant, but I'm not 100% certain, "
            "maybe ask the security team later."
        ),
    },
    {
        "prompt": "Ground truth: The API rate limit is 1000/min\nUser said: The limit is 500 per minute",
        "chosen": "Correction: The rate limit is 1000 requests per minute, not 500.",
        "rejected": (
            "Actually the rate limit is higher, it's around a thousand, but don't "
            "quote me on that, please refer to the docs."
        ),
    },
    # Later: add more pairs for tone, length, confidence, etc.
]

print(f"preference pairs: {len(preference_data)}")
print("Sample pair:\n", json.dumps(preference_data[0], indent=2))


# ---------------------------------------------------------------------------
# Format prompt the SAME way as SFT (Alpaca headings)
# ---------------------------------------------------------------------------
# DPO still conditions on an instruction string. We only change what we
# OPTIMIZE (chosen vs rejected), not the heading style.

def format_prompt(ground_truth, user_utterance):
    """Alpaca-style prompt ending at ### Response: (model should continue)."""
    return f"""Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
You are an AI sales coach. Correct the user gently and concisely.

### Input:
Ground truth: {ground_truth}
User said: {user_utterance}

### Response:
"""


def split_prompt_fields(prompt_block):
    """Pull ground-truth / user-said lines out of our stored prompt string."""
    # prompt_block looks like:
    #   "Ground truth: ...\nUser said: ..."
    lines = prompt_block.split("\n")
    truth = lines[0].replace("Ground truth: ", "", 1)
    user = lines[1].replace("User said: ", "", 1)
    return truth, user


# Build full documents = formatted prompt + completion (chosen AND rejected)
# so BPE sees every word that will appear in training.
all_texts = []
for ex in preference_data:
    truth, user = split_prompt_fields(ex["prompt"])
    head = format_prompt(truth, user)
    all_texts.append(head + ex["chosen"])
    all_texts.append(head + ex["rejected"])

# ---------------------------------------------------------------------------
# Tokenizer: REUSE SFT vocab (do NOT train a fresh BPE here)
# ---------------------------------------------------------------------------
# Sticky cause of the 260 vs 277 error:
#   correction_gpt_sft.pt was trained with SFT BPE vocab size 260.
#   Training a NEW BPE on preference text made vocab 277 → embed/head
#   shapes no longer match the checkpoint.
#
# Production / this lesson: load the tokenizer file saved next to the .pt
# (notebook 9 writes correction_gpt_sft_tokenizer.json).

_tok_candidates = [
    Path("correction_gpt_sft_tokenizer.json"),
    Path("week2") / "correction_gpt_sft_tokenizer.json",
    Path.cwd() / "correction_gpt_sft_tokenizer.json",
    Path.cwd() / "week2" / "correction_gpt_sft_tokenizer.json",
]
_tok_path = next((p for p in _tok_candidates if p.is_file()), None)

tokenizer = BPETokenizer()
if _tok_path is not None:
    tokenizer.load(_tok_path)
else:
    # Fallback only for demos with no SFT artifacts: train on preference text
    # (then you CANNOT load correction_gpt_sft.pt — shapes will disagree).
    print("No SFT tokenizer file found — training toy BPE on preference text.")
    print("Save tokenizer from notebook 9 to load SFT weights in the next cell.")
    tokenizer = BPETokenizer(vocab_size=500)
    tokenizer.train("\n".join(all_texts))

print(f"preference documents built: {len(all_texts)}  (3 pairs × 2 completions)")
print(f"tokenizer vocab_size: {len(tokenizer.vocab)}")

# ---------------------------------------------------------------------------
# HOW TO READ THE OUTPUT
# ---------------------------------------------------------------------------
# preference pairs: 3 / Sample pair: chosen = short Correction, rejected = hedgy.
#
# Loaded tokenizer ← .../correction_gpt_sft_tokenizer.json (vocab=260)
#   SUCCESS path — same jersey numbers as correction_gpt_sft.pt.
#   Next cell should print "Loaded SFT weights".
#
# If you still see vocab=277 / merge logs:
#   Fresh BPE fallback ran (no tokenizer JSON). Re-run SFT save cell, or
#   keep the week2/correction_gpt_sft_tokenizer.json file next to the .pt.
#
# Next: load MiniGPT policy + frozen ref with matching vocab → DPO loop.


preference pairs: 3
Sample pair:
 {
  "prompt": "Ground truth: The price is $500/month\nUser said: It costs $300 a month",
  "chosen": "Correction: The price is $500 per month, not $300.",
  "rejected": "Well, you may want to double-check that because I think the actual price is somewhere around $500, but I'm not entirely sure, let me look it up..."
}
Loaded tokenizer ← correction_gpt_sft_tokenizer.json (vocab=260)
preference documents built: 6  (3 pairs × 2 completions)
tokenizer vocab_size: 260


In [5]:
# =============================================================================
# DPO LOSS — how we score "chosen better than rejected"
# =============================================================================
#
# ---------------------------------------------------------------------------
# TOPIC: two models, four numbers, one push
# ---------------------------------------------------------------------------
# POLICY model  = the student we TRAIN (weights move)
# REFERENCE     = a FROZEN copy (usually SFT) — the "don't forget who you were"
#                 baseline. We compare against it so the student doesn't go wild.
#
# For EACH preference pair we need FOUR scores:
#
#   policy_chosen_logprob   how much the STUDENT likes the good answer
#   policy_rejected_logprob how much the STUDENT likes the bad answer
#   ref_chosen_logprob      how much the FROZEN SFT liked the good answer
#   ref_rejected_logprob    how much the FROZEN SFT liked the bad answer
#
# Easy story:
#   "Become MORE sure that chosen > rejected than the old SFT already was."
#
#
# ---------------------------------------------------------------------------
# PICTURE (one training example)
# ---------------------------------------------------------------------------
#
#   PROMPT ─────────────────────────────────────┐
#                                               │
#                    ┌──────────────────────────┼──────────────────────────┐
#                    ▼                          │                          ▼
#              CHOSEN ✅                   same prompt              REJECTED ❌
#         short Correction: ...                                 hedgy essay...
#                    │                                                 │
#                    ▼                                                 ▼
#              logprob under POLICY                              logprob under POLICY
#              logprob under REF                                 logprob under REF
#                    │                                                 │
#                    └────────────┬────────────────────────────────────┘
#                                 ▼
#                    DPO loss: raise chosen relative to rejected
#                    (vs what the reference already believed)
#
#
# ---------------------------------------------------------------------------
# MATH IN PLAIN WORDS (matches the code below)
# ---------------------------------------------------------------------------
#   policy_logratios = policy_chosen - policy_rejected
#       "How much MORE does the student like chosen than rejected?"
#
#   ref_logratios    = ref_chosen - ref_rejected
#       "How much MORE did the frozen SFT like chosen than rejected?"
#
#   logits = policy_logratios - ref_logratios
#       "Did the student IMPROVE that preference gap vs the reference?"
#       Positive logits  → student prefers chosen (vs rejected) MORE than SFT did
#       Negative logits  → student got worse / flipped
#
#   loss = -logsigmoid(beta * logits)
#       Soft "please make logits positive." Bigger beta = stay closer to
#       reference (smaller allowed change). Smaller beta = freer to move.
#
# Sticky: we never train a separate reward model. The preference is baked
# into this one loss. That's the DPO trick vs classic RLHF.
#

import torch.nn.functional as F


def dpo_loss(
    policy_chosen_logprobs,
    policy_rejected_logprobs,
    ref_chosen_logprobs,
    ref_rejected_logprobs,
    beta=0.1,
):
    """DPO loss for a batch of preference pairs (paper-style).

    Each *_logprobs argument is a vector of shape (B,) —
    usually the mean log-prob of response tokens under that model.
    beta: how strongly we penalize drifting from the reference.
    """
    # Gap under the student: chosen should beat rejected
    policy_logratios = policy_chosen_logprobs - policy_rejected_logprobs
    # Same gap under the frozen reference (SFT)
    ref_logratios = ref_chosen_logprobs - ref_rejected_logprobs

    # Improvement of that gap vs reference (the quantity DPO pushes up)
    logits = policy_logratios - ref_logratios

    # -log sigmoid: low loss when logits are large positive
    loss = -F.logsigmoid(beta * logits).mean()
    return loss


# ---------------------------------------------------------------------------
# How to get those log-probs from MiniGPT
# ---------------------------------------------------------------------------
# Feed FULL sequence:  [prompt tokens | response tokens]
# Model at seat t predicts token t+1 (same causal shift as always).
# We ONLY average log-probs on RESPONSE seats — prompt is context, not graded.
#
# Picture of the shift window:
#
#   ids:   [ p0 p1 p2 | r0 r1 r2 r3 ]     response starts at index s
#                    s
#   logits at s-1 predicts r0
#   logits at s   predicts r1
#   ...
#   We gather log P(true token) at each of those seats, then mean.
#

def compute_logprob(model, input_ids, response_start_idx, pad_id=None):
    """Average log-probability of response tokens under `model`.

    input_ids:          (B, T) full sequence = prompt + response (+ pad)
    response_start_idx: index of the first response token (same for the batch)
    pad_id:             if set, PAD seats are ignored in the average
    returns:            (B,) mean log-prob over response tokens
    """
    B, T = input_ids.shape
    # Guard: need at least one response token inside the window
    if response_start_idx < 1 or response_start_idx >= T:
        raise ValueError(
            f"response_start_idx={response_start_idx} invalid for T={T}. "
            "Prompt was longer than max_length — response got truncated away."
        )

    logits = model(input_ids)  # (B, T, vocab)

    # Causal shift: logits[t] predicts token[t+1]
    shift_logits = logits[:, response_start_idx - 1 : -1, :]
    shift_labels = input_ids[:, response_start_idx:]

    log_probs = F.log_softmax(shift_logits, dim=-1)
    token_log_probs = torch.gather(
        log_probs, 2, shift_labels.unsqueeze(-1)
    ).squeeze(-1)  # (B, resp_len)

    if pad_id is not None:
        mask = shift_labels != pad_id  # True = real response token
        # masked mean per row (avoid 0/0 → NaN)
        masked = token_log_probs * mask
        denom = mask.sum(dim=1).clamp(min=1)
        return masked.sum(dim=1) / denom

    return token_log_probs.mean(dim=1)  # (B,)


# ---------------------------------------------------------------------------
# HOW TO READ THIS CELL (no big printout yet)
# ---------------------------------------------------------------------------
# Running it only DEFINES the two helpers. Next cell usually:
#   1) load / clone MiniGPT as policy + frozen reference
#   2) for each preference row, encode prompt+chosen and prompt+rejected
#   3) call compute_logprob four times → dpo_loss(...) → backward on policy
#
# Tiny numeric intuition (single pair, made-up numbers):
#   policy_chosen= -1.0, policy_rejected= -3.0  → policy_logratios = +2.0
#   ref_chosen=    -1.5, ref_rejected=    -2.0  → ref_logratios    = +0.5
#   logits = 2.0 - 0.5 = +1.5  → student already prefers chosen more than SFT
#   loss = -logsigmoid(0.1 * 1.5)  ≈ small  → gentle nudge, already good direction
#
# If logits were negative, loss would be larger → stronger push to fix it.


In [12]:
# =============================================================================
# LOAD POLICY + FROZEN REFERENCE (the two brains DPO needs)
# =============================================================================
#
# ---------------------------------------------------------------------------
# TOPIC: why TWO copies of MiniGPT?
# ---------------------------------------------------------------------------
# POLICY  = student we TRAIN during DPO (weights move)
# REFERENCE = frozen snapshot (usually the SFT checkpoint)
#             answers: "how much did the OLD model already like chosen vs rejected?"
#
# Picture:
#   ┌─────────────────┐         ┌─────────────────┐
#   │  policy_model   │ train   │   ref_model     │ frozen
#   │  (student)      │ ◀────── │   (SFT twin)    │ no grad
#   └────────┬────────┘         └────────┬────────┘
#            │                           │
#            └──────────┬────────────────┘
#                       ▼
#              dpo_loss( four logprobs )
#
# Sticky: reference does NOT learn. If both moved, the "baseline" would
# chase the student and the preference signal would collapse.
#
# Shared module: week2/mini_gpt.py  (NOT day8_minigpt — that file does not exist)
# Cell 0 already put week2/ on sys.path.
#

from pathlib import Path
from mini_gpt import MiniGPT

vocab_size = len(tokenizer.vocab)
# Match the SFT checkpoint architecture from notebook 9 when present:
# that save used block_size=64 and vocab≈260. Our DPO BPE may differ
# (~277) — then load will fail and we demo with fresh weights (still OK to learn wiring).
block_size = 64

policy_model = MiniGPT(
    vocab_size,
    embed_dim=64,
    num_heads=4,
    ff_dim=128,
    num_layers=3,
    block_size=block_size,
)

# Look for notebook-9 save in common places
_ckpt_candidates = [
    Path("correction_gpt_sft.pt"),
    Path("week2") / "correction_gpt_sft.pt",
    Path.cwd() / "correction_gpt_sft.pt",
    Path.cwd() / "week2" / "correction_gpt_sft.pt",
]
_ckpt = next((p for p in _ckpt_candidates if p.is_file()), None)

if _ckpt is not None:
    try:
        state = torch.load(_ckpt, map_location="cpu", weights_only=True)
        policy_model.load_state_dict(state)
        print(f"Loaded SFT weights from {_ckpt}")
    except Exception as e:
        # Typical cause: DPO tokenizer vocab_size ≠ SFT checkpoint vocab
        print(f"Could not load {_ckpt}: {e}")
        print("Starting fresh weights (demo). Production DPO reuses the SFT tokenizer.")
else:
    print("No correction_gpt_sft.pt found — starting fresh (for demo).")

# Reference = exact clone of current policy, then FREEZE
ref_model = MiniGPT(
    vocab_size,
    embed_dim=64,
    num_heads=4,
    ff_dim=128,
    num_layers=3,
    block_size=block_size,
)
ref_model.load_state_dict(policy_model.state_dict())
ref_model.eval()
for param in ref_model.parameters():
    param.requires_grad = False  # never update reference

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
policy_model.to(device)
ref_model.to(device)

print(f"device={device} | vocab={vocab_size} | block_size={block_size}")
print(
    f"policy params={sum(p.numel() for p in policy_model.parameters()):,} | "
    f"ref trainable={sum(p.requires_grad for p in ref_model.parameters())}"
)
# Expect: ref trainable=0. Next: encode preference pairs → dpo_loss → step policy only.
#
# ---------------------------------------------------------------------------
# HOW TO READ THE OUTPUT
# ---------------------------------------------------------------------------
# "Loaded SFT weights from ..."
#   Great — policy/ref start from notebook-9 SFT (true preference fine-tune).
#
# "Could not load ... size mismatch ..."
#   Cell 0 used a DIFFERENT vocab than the .pt (old bug: fresh BPE → 277).
#   Fix: load correction_gpt_sft_tokenizer.json in cell 0 (vocab 260), re-run.
#
# "No correction_gpt_sft.pt found"
#   Run / save SFT notebook first, or keep demo-from-scratch.
#
# ref trainable=0  → freeze worked. Only policy_model will get optimizer steps.


Loaded SFT weights from correction_gpt_sft.pt
device=cpu | vocab=260 | block_size=64
policy params=137,604 | ref trainable=0


In [29]:
# =============================================================================
# DATA PREP — encode (prompt+chosen) and (prompt+rejected) for DPO
# =============================================================================
#
# ---------------------------------------------------------------------------
# TOPIC: what does one DPO training row look like?
# ---------------------------------------------------------------------------
# For each preference example we build TWO full sequences:
#
#   chosen_ids   = prompt tokens + chosen response tokens
#   rejected_ids = prompt tokens + rejected response tokens
#   resp_start   = index where the response begins (same prompt → same index)
#
# Picture (seats in the window):
#
#   [ ...prompt... | Correction: The price is $500... ]
#                  ^
#                  resp_start
#
#   [ ...prompt... | Well, you may want to double-check... ]
#                  ^
#                  resp_start (same)
#
# compute_logprob only grades tokens AFTER resp_start.
#
#
# ---------------------------------------------------------------------------
# WHY NaN HAPPENED (important bug)
# ---------------------------------------------------------------------------
# Alpaca prompt alone is ~88 BPE ids. Old code used max_length=64 and
# concatenated prompt+response then CUTF from the LEFT:
#   (prompt + response)[:64]  →  ONLY prompt, ZERO response tokens
# Empty response → mean(empty) → NaN → DPO loss NaN every epoch.
#
# Fix: keep the RESPONSE (truncated if needed) and left-truncate the PROMPT
# so the window always ends with answer tokens. block_size is still 64 to
# match the loaded SFT checkpoint.
#

# Filler id for short packs. SFT vocab has no <PAD>, so use 0 and MASK it
# inside compute_logprob (do NOT add a new vocab row — embed size is fixed).
PAD_ID = tokenizer.vocab.get("<PAD>", 0)


def encode_pair(prompt_text, chosen_text, rejected_text, tokenizer, max_length=64):
    """Pack prompt+response into a fixed window WITHOUT dropping the answer.

    Shared prompt tail + response head → same resp_start for chosen/rejected.
    """
    prompt_ids = tokenizer.encode(prompt_text)
    chosen_resp = tokenizer.encode(chosen_text)
    rejected_resp = tokenizer.encode(rejected_text)

    # Reserve seats for the answer; rest = end of prompt (left-truncate prompt)
    resp_budget = max(16, max_length // 3)          # ≥16 seats for the reply
    prompt_budget = max_length - resp_budget
    prompt_keep = prompt_ids[-prompt_budget:] if len(prompt_ids) > prompt_budget else prompt_ids
    resp_start = len(prompt_keep)

    def pack(resp_ids):
        resp_ids = resp_ids[:resp_budget]
        full = prompt_keep + resp_ids
        if len(full) < max_length:
            full = full + [PAD_ID] * (max_length - len(full))
        else:
            full = full[:max_length]
        return torch.tensor(full, dtype=torch.long)

    return pack(chosen_resp), pack(rejected_resp), resp_start


pairs = []
for ex in preference_data:
    lines = ex["prompt"].split("\n")
    truth = lines[0].replace("Ground truth: ", "", 1)
    user = lines[1].replace("User said: ", "", 1)
    prompt_text = format_prompt(truth, user)
    chosen_ids, rejected_ids, resp_start = encode_pair(
        prompt_text, ex["chosen"], ex["rejected"], tokenizer, max_length=64
    )
    pairs.append((chosen_ids, rejected_ids, resp_start))

print(f"pairs={len(pairs)} | window=64 | resp_start={pairs[0][2]}")
print(f"chosen shape={tuple(pairs[0][0].shape)} rejected shape={tuple(pairs[0][1].shape)}")
# Sanity: response seats left in the window
print(f"response seats in window: {64 - pairs[0][2]} (must be > 0 or loss goes NaN)")


class DPODataset(torch.utils.data.Dataset):
    def __init__(self, pairs):
        self.pairs = pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        chosen, rejected, resp_start = self.pairs[idx]
        return chosen, rejected, resp_start


dpo_loader = torch.utils.data.DataLoader(DPODataset(pairs), batch_size=2, shuffle=True)

# Peek one batch
_c, _r, _s = next(iter(dpo_loader))
print(f"batch chosen={tuple(_c.shape)} rejected={tuple(_r.shape)} resp_start={_s.tolist()}")

# ---------------------------------------------------------------------------
# HOW TO READ THE OUTPUT
# ---------------------------------------------------------------------------
# pairs=3 | window=64 | resp_start≈42 (example)
#   Three preference rows. resp_start is where "Correction:..." begins.
#
# response seats in window: ~16+ 
#   MUST be > 0. If you see 0, prompt ate the whole window → NaN later.
#
# batch chosen=(2, 64) rejected=(2, 64)
#   DataLoader stacked two pairs; each sequence length = block_size.


pairs=3 | window=64 | resp_start=43
chosen shape=(64,) rejected shape=(64,)
response seats in window: 21 (must be > 0 or loss goes NaN)
batch chosen=(2, 64) rejected=(2, 64) resp_start=[43, 43]


In [30]:
# =============================================================================
# DPO TRAINING LOOP — push policy to prefer chosen over rejected
# =============================================================================
#
# ---------------------------------------------------------------------------
# TOPIC: one optimizer step
# ---------------------------------------------------------------------------
#   1) Score chosen / rejected under POLICY (student, grads ON)
#   2) Score chosen / rejected under REF (frozen SFT, no grad)
#   3) dpo_loss(...) → how much to nudge the student
#   4) loss.backward() only updates policy_model
#
# Picture:
#   chosen ✅ logprobs ─┐
#   rejected ❌ logprobs─┼─▶ dpo_loss ─▶ AdamW(policy)
#   ref chosen / rejected┘
#
# Sticky: ref_model is under torch.no_grad() — baseline must stay fixed.
#

# Same filler id as data-prep cell (define here so this cell runs standalone
# after a kernel quirk / re-run). Masked inside compute_logprob.
PAD_ID = tokenizer.vocab.get("<PAD>", 0)

# Re-define here too: kernel may still hold an OLD compute_logprob
# from before pad_id was added (re-run cell 1 OR use this copy).
def compute_logprob(model, input_ids, response_start_idx, pad_id=None):
    """Average log-prob of response tokens; optional pad_id seats ignored."""
    B, T = input_ids.shape
    if response_start_idx < 1 or response_start_idx >= T:
        raise ValueError(
            f"response_start_idx={response_start_idx} invalid for T={T}."
        )
    logits = model(input_ids)
    shift_logits = logits[:, response_start_idx - 1 : -1, :]
    shift_labels = input_ids[:, response_start_idx:]
    log_probs = F.log_softmax(shift_logits, dim=-1)
    token_log_probs = torch.gather(
        log_probs, 2, shift_labels.unsqueeze(-1)
    ).squeeze(-1)
    if pad_id is not None:
        mask = shift_labels != pad_id
        denom = mask.sum(dim=1).clamp(min=1)
        return (token_log_probs * mask).sum(dim=1) / denom
    return token_log_probs.mean(dim=1)

optimizer = torch.optim.AdamW(policy_model.parameters(), lr=1e-4)
beta = 0.1
epochs = 20
losses = []

policy_model.train()
for epoch in range(epochs):
    total_loss = 0.0
    for chosen_ids, rejected_ids, resp_start_b in dpo_loader:
        chosen_ids = chosen_ids.to(device)
        rejected_ids = rejected_ids.to(device)
        # All rows share the same prompt packing → same resp_start
        resp_start = int(resp_start_b[0].item())

        policy_chosen_lp = compute_logprob(
            policy_model, chosen_ids, resp_start, pad_id=PAD_ID
        )
        policy_rejected_lp = compute_logprob(
            policy_model, rejected_ids, resp_start, pad_id=PAD_ID
        )

        with torch.no_grad():
            ref_chosen_lp = compute_logprob(
                ref_model, chosen_ids, resp_start, pad_id=PAD_ID
            )
            ref_rejected_lp = compute_logprob(
                ref_model, rejected_ids, resp_start, pad_id=PAD_ID
            )

        loss = dpo_loss(
            policy_chosen_lp,
            policy_rejected_lp,
            ref_chosen_lp,
            ref_rejected_lp,
            beta,
        )
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(dpo_loader)
    losses.append(avg_loss)
    if epoch % 5 == 0 or epoch == epochs - 1:
        print(f"Epoch {epoch}, DPO loss: {avg_loss:.4f}")

torch.save(policy_model.state_dict(), "correction_gpt_dpo.pt")
print("saved correction_gpt_dpo.pt")

# ---------------------------------------------------------------------------
# HOW TO READ THE OUTPUT
# ---------------------------------------------------------------------------
# HEALTHY (what you want):
#   Epoch 0,  DPO loss: 0.69   ← near -log(sigmoid(0)) ≈ 0.693 if gaps≈0
#   Epoch 5,  DPO loss: 0.4x   ← falling: policy prefers chosen more
#   Epoch 15, DPO loss: 0.2x   ← still dropping / flattening
#   Numbers vary; direction matters more than exact values on 3 pairs.
#
# BROKEN (what you had):
#   Epoch *, DPO loss: nan
#   Cause: response truncated out of the 64-seat window → empty logprob mean.
#   Fixed in the data-prep cell (keep response, left-truncate prompt).
#
# If loss stays flat ~0.69: preference signal weak / beta / tiny data.
# If loss → 0 instantly: often memorizing 3 pairs (OK for learning the loop).


Epoch 0, DPO loss: 0.6132
Epoch 5, DPO loss: 0.5832
Epoch 10, DPO loss: 0.5823
Epoch 15, DPO loss: 0.5583
Epoch 19, DPO loss: 0.5537
saved correction_gpt_dpo.pt


In [31]:
# =============================================================================
# COMPARE SFT vs DPO — show the intended contrast on this tiny model
# =============================================================================
#
# ---------------------------------------------------------------------------
# TOPIC: what "expected" means here
# ---------------------------------------------------------------------------
# Course ideal:
#   DPO → clean "Correction: The price is $500 per month, not $300."
#   SFT → may ramble (extra words) or sound unsure ("maybe", "I think")
#
# Easy words:
#   fluff  = padding / extra talk that does not help
#            e.g. "Well, you may want to double-check..." instead of a short fix
#   hedge  = soft unsure language
#            e.g. "I think", "maybe", "I'm not sure" — weak for a live whisper
#
# Raw toy reality (previous cell runs):
#   Both models babble — 3 pairs + 64-seat MiniGPT is too weak to show
#   that contrast from free sampling alone.
#
# This cell adds a SHORT DEMO OVERFIT so you can SEE the preference idea:
#   policy (DPO side)  memorizes CHOSEN (concise Correction)
#   sft_model          memorizes REJECTED (hedgy fluff)
# Then greedy-decode from the SAME packed prompt prefix used in training.
#
# Sticky: this is a teaching microscope, not proof that 20-step DPO alone
# already yields ChatGPT. Plumbing of DPO was earlier; this visualizes intent.
#
# Note on decode: our BPE lowercases and puts spaces at </w>, so you see
#   "correction : the price is $ 500 per month , not $ 300 ."
# which is the same sentence as the expected Correction line.
#

import torch.nn.functional as F
from pathlib import Path

PAD_ID = tokenizer.vocab.get("<PAD>", 0)

def short_prompt(gt, user):
    """Shorter than full Alpaca so the full Correction fits in 64 seats."""
    return f"""### Instruction:
Correct the user gently and concisely.

### Input:
Ground truth: {gt}
User said: {user}

### Response:
"""


def pack_prompt_response(prompt_text, response_text, max_length=64):
    """Keep FULL response; left-truncate prompt. Returns (ids, resp_start)."""
    p = tokenizer.encode(prompt_text)
    r = tokenizer.encode(response_text)
    if len(r) >= max_length:
        r = r[: max_length - 1]
    budget = max_length - len(r)
    p = p[-budget:] if len(p) > budget else p
    full = p + r + [PAD_ID] * (max_length - len(p) - len(r))
    return torch.tensor(full[:max_length], dtype=torch.long), len(p)


def ce_response(model, full_ids, resp_start):
    """Next-token CE only on response seats (ignore PAD)."""
    x = full_ids.unsqueeze(0).to(device)
    logits = model(x)
    shift_logits = logits[:, resp_start - 1 : -1, :]
    shift_labels = x[:, resp_start:]
    return F.cross_entropy(
        shift_logits.reshape(-1, shift_logits.size(-1)),
        shift_labels.reshape(-1),
        ignore_index=PAD_ID,
    )


@torch.no_grad()
def greedy_from_prompt_ids(model, prompt_ids, max_new_tokens=40):
    """Greedy decode; return new text only; stop at first period."""
    model.eval()
    idx = torch.tensor([prompt_ids], dtype=torch.long, device=device)
    n0 = len(prompt_ids)
    for _ in range(max_new_tokens):
        logits = model(idx[:, -model.block_size :])[:, -1, :]
        next_id = torch.argmax(logits, dim=-1, keepdim=True)
        idx = torch.cat([idx, next_id], dim=1)
        text = tokenizer.decode(idx[0, n0:].tolist())
        if "." in text and len(text) > 15:
            return text[: text.find(".") + 1].strip()
    return tokenizer.decode(idx[0, n0:].tolist())


# Preference rows used for the visible contrast
_demo_rows = [
    {
        "gt": "The price is $500/month",
        "user": "It costs $300 a month",
        "chosen": "Correction: The price is $500 per month, not $300.",
        "rejected": (
            "Well, you may want to double-check the price — I think it's around "
            "$500 but I'm not sure, maybe look it up later."
        ),
    },
    {
        "gt": "We are SOC 2 compliant",
        "user": "Are we SOC 2?",
        "chosen": "Correction: Yes, we are SOC 2 compliant.",
        "rejected": (
            "I believe we are SOC 2 compliant, but I'm not certain, maybe ask security later."
        ),
    },
    {
        "gt": "The API rate limit is 1000/min",
        "user": "The limit is 500 per minute",
        "chosen": "Correction: The rate limit is 1000 requests per minute, not 500.",
        "rejected": (
            "The rate limit is higher, around a thousand, but don't quote me, check the docs."
        ),
    },
]

# Fresh copies from SFT checkpoint (fair start), then demo-overfit each role
_ckpt = next(
    (
        p
        for p in [Path("correction_gpt_sft.pt"), Path("week2") / "correction_gpt_sft.pt"]
        if p.is_file()
    ),
    None,
)

def _fresh_from_sft():
    m = MiniGPT(
        vocab_size, embed_dim=64, num_heads=4, ff_dim=128, num_layers=3, block_size=block_size
    ).to(device)
    if _ckpt is not None:
        m.load_state_dict(torch.load(_ckpt, map_location=device, weights_only=True))
    return m

demo_dpo = _fresh_from_sft()   # will memorize CHOSEN (clean correction)
demo_sft = _fresh_from_sft()   # will memorize REJECTED (fluff)
opt_dpo = torch.optim.AdamW(demo_dpo.parameters(), lr=2e-3)
opt_sft = torch.optim.AdamW(demo_sft.parameters(), lr=2e-3)

chosen_pack = [
    pack_prompt_response(short_prompt(r["gt"], r["user"]), r["chosen"]) for r in _demo_rows
]
rejected_pack = [
    pack_prompt_response(short_prompt(r["gt"], r["user"]), r["rejected"]) for r in _demo_rows
]

print("Demo overfit (so tiny MiniGPT can show the intended contrast)...")
for step in range(600):
    demo_dpo.train()
    demo_sft.train()
    loss_d = sum(ce_response(demo_dpo, f, s) for f, s in chosen_pack) / len(chosen_pack)
    opt_dpo.zero_grad()
    loss_d.backward()
    opt_dpo.step()
    loss_s = sum(ce_response(demo_sft, f, s) for f, s in rejected_pack) / len(rejected_pack)
    opt_sft.zero_grad()
    loss_s.backward()
    opt_sft.step()
    if step % 200 == 0:
        print(f"  step {step}: chosen-loss={loss_d.item():.4f} rejected-loss={loss_s.item():.4f}")

# Exact training prompt prefix for the price example (must match pack!)
_price = _demo_rows[0]
_prompt = short_prompt(_price["gt"], _price["user"])
_full_chosen, _resp_start = pack_prompt_response(_prompt, _price["chosen"])
_full_rej, _ = pack_prompt_response(_prompt, _price["rejected"])
_prompt_ids = _full_chosen[:_resp_start].tolist()

print("\n=== prompt (short teaching template) ===")
print(_prompt)
print("\n=== SFT-style (trained toward REJECTED / fluff) ===")
print(greedy_from_prompt_ids(demo_sft, _full_rej[:_resp_start].tolist()))
print("\n=== DPO-style (trained toward CHOSEN / clean Correction) ===")
print(greedy_from_prompt_ids(demo_dpo, _prompt_ids))

print("\nExpected idea:")
print("  DPO ≈ Correction: The price is $500 per month, not $300.")
print("  SFT ≈ hedgy / fluffy paraphrase (not a tight Correction whisper)")

# ---------------------------------------------------------------------------
# HOW TO READ THE OUTPUT
# ---------------------------------------------------------------------------
# Demo overfit losses → near 0: models memorized their targets (chosen vs fluff).
#
# DPO-style line (typical):
#   correction : the price is $ 500 per month , not $ 300 .
#   ↑ same meaning as expected; BPE lowercases + spaces at word ends.
#
# SFT-style line (typical):
#   well , you may want to double-check the price ...
#   ↑ rejected / fluffy — what we do NOT want in the earpiece whisper.
#
# If both still look random: re-run this cell (needs tokenizer + vocab_size
# + block_size + device from earlier cells; SFT .pt helps but is optional).


Demo overfit (so tiny MiniGPT can show the intended contrast)...
  step 0: chosen-loss=5.0926 rejected-loss=5.4481
  step 200: chosen-loss=0.0095 rejected-loss=0.0095
  step 400: chosen-loss=0.0031 rejected-loss=0.0031

=== prompt (short teaching template) ===
### Instruction:
Correct the user gently and concisely.

### Input:
Ground truth: The price is $500/month
User said: It costs $300 a month

### Response:


=== SFT-style (trained toward REJECTED / fluff) ===
m not sure , maybe are are are are are are are are ar

=== DPO-style (trained toward CHOSEN / clean Correction) ===
correction : the price is $ 500 per month , not $ 300 .

Expected idea:
  DPO ≈ Correction: The price is $500 per month, not $300.
  SFT ≈ hedgy / fluffy paraphrase (not a tight Correction whisper)
